## Multi-Head Attention

入力 $(batch, seq, d_{model})$ を複数のヘッドに分けて Scaled Dot-Product Attention を並列に計算し、まとめ直す。出力の形は入力と同じ。

$$\text{head}_i = \text{softmax}\!\left(\frac{Q_i K_i^\top}{\sqrt{d_k}}\right)V_i$$

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1,\dots,\text{head}_h)\,W_o$$

### 形状

| 記号 | 形状 |
|---|---|
| $Q,K,V$ 入力 | $(batch, seq, d_{model})$ |
| $W_q, W_k, W_v, W_o$ | $(d_{model}, d_{model})$ |
| ヘッド分割後 | $(batch, num\_heads, seq, d_k)$ |
| $d_k$ | $d_{model} / num\_heads$ |

### 流れ

$$x \xrightarrow{W_q,W_k,W_v} Q,K,V \xrightarrow{reshape + transpose} \text{ヘッド分割} \xrightarrow{\text{Attention}} \text{context} \xrightarrow{reshape} W_o \rightarrow \text{out}$$

- $W_q, W_k, W_v$: 入力を $Q, K, V$ に射影する($d_{model}$ のまま)
- reshape + transpose: $d_{model}$ を $num\_heads$ 個の $d_k$ に分ける
- $W_o$: 結合したヘッドを混ぜて出力する


In [3]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import math

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_modelはnum_headで割り切れる必要があります"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        #線形変換の道具　マルチヘッドだけど実装ではまずは次元を落とさずに射影
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        #出力用にまとめる
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, Q_input, K_input, V_input, mask=None):
        # 入力: Q, K, V.shape = (batch, seq, d_model)
        batch_size = Q_input.size(0)

        #線形変換　実行
        Q = self.W_q(Q_input)   # (batch, seq, d_model)
        K = self.W_k(K_input)   
        V = self.W_v(V_input)   # (batch, seq, d_model)

        #ヘッドを分割
        # reshapeで(batch, seq, d_model)　→ (batch, seq, num_heads, d_k)
        # transpose(1, 2)で(batch, seq, num_heads, d_k) → (batch, num_heads, seq, d_k)
        Q = Q.reshape(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)   
        K = K.reshape(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)   
        V = V.reshape(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)   # (batch, num_heads, seq, d_k)

        # Scaled Dot-Product Attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)   # (batch, num_heads, seq, seq)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))             
        attention_weights = F.softmax(scores, dim=-1) #scoresを正規化

        # attention_weights @ V: (batch, num_heads, seq, seq) @ (batch, num_heads, seq, d_k) = (batch, num_heads, seq, d_k)
        context = torch.matmul(attention_weights, V)   # (batch, num_heads, seq, d_k)

        #ヘッドを結合
        #(batch, num_heads, seq, d_k)　→　(batch, seq, num_heads, d_k) →　(batch, seq, d_model)
        context = context.transpose(1, 2).reshape(batch_size, -1, self.d_model)

        #出力
        output = self.W_o(context)
        return output

        

In [5]:
#テスト
d_model = 512
num_heads = 8
mha = MultiHeadAttention(d_model, num_heads)


# ダミー入力: (batch=2, seq_len=10, d_model=512)
x = torch.randn(2, 10, 512)
output = mha(x, x, x)  # Self-Attentionの場合、Q_input = K_input = V_input = x 
print("入力形状:", x.shape)
print("出力形状:", output.shape)
print("形状が一致:", x.shape == output.shape)

入力形状: torch.Size([2, 10, 512])
出力形状: torch.Size([2, 10, 512])
形状が一致: True


In [ ]:
# 想定: 文章0 は本物7単語+PAD3つ、文章1 は本物5単語+PAD5つ
mask = torch.tensor([
    [1, 1, 1, 1, 1, 1, 1, 0, 0, 0],   # 文章0: 後ろ3つを無視
    [1, 1, 1, 1, 1, 0, 0, 0, 0, 0],   # 文章1: 後ろ5つを無視
])   # (2, 10)

# scores の形 (batch, num_heads, seq, seq) に合うように整形
mask = mask[:, None, None, :]   # (2, 10) → (2, 1, 1, 10)

# mask あり/なし両方走らせて比較
output_no_mask  = mha(x, x, x)
output_masked   = mha(x, x, x, mask)

print("マスクなし出力:", output_no_mask.shape)
print("マスクあり出力:", output_masked.shape)
print("出力が異なる:", not torch.allclose(output_no_mask, output_masked)) 
#torch.allclose(a, b) = 「a と b が すべての要素でほぼ等しい か?」を判定する


マスクなし出力: torch.Size([2, 10, 512])
マスクあり出力: torch.Size([2, 10, 512])
出力が異なる: True


## 詰まったところ
- 理論上は各ヘッドごとに別々の重み行列を持つけど実装ではまず次元を落とさずに大きな線形層で射影してその圧reshapeでヘッドに分割すること<br>
→行列積は「右の行列の列ごとに独立」だから。よって複数の小さい行列を横に連結して、1回の大きな行列積で同時に計算できる

- .transpose(1, 2)をする意味
→attention の計算式は Q @ K^T。
これを「seq 個のトークン同士の類似度行列(seq × seq)を作る計算」にしたい。
行列積のルールから、(seq, d_k) @ (d_k, seq) = (seq, seq) になる組み合わせが必要だから


-  F.softmax(scores, dim=1)のdim=-1の使い方<br>
→最後の次元で正規化　今回は shape(scores)=(〇, 〇) だから列方向（横向き）に正規化


- 重み行列（w_q, W_k, W_v）には同じ行列（入力トークンの塊）を入れると思っていたが、コードを見ると入力からQ, K, Vに分けて入れていること<br>
→同じものをを入れるのはSelf-Attentionの場合で Cross-Attention では、クエリの元になる入力と、キー・バリューの元になる入力が、もともと別物のデータ（例：翻訳での訳文側 と 原文側）。1つの引数では2種類のデータを渡せない。だから別々の引数を渡す必要がある。


- `x = torch.randn(2, 10, 512)` のように、batch 内の全文章は同じ seq に揃っている前提になっている。なぜ揃える必要があるのか<br>
→ そもそも複数の文を１つのテンソルにしたい。そしてテンソルは行ごとに長さが違うと作れない(`[[1,2,3], [4,5]]` のような形は不可)。本来文章ごとに単語数はバラバラなので、短い文章の後ろに PAD トークンを足して長さを揃える処理(padding)が必要になる。padding は MultiHeadAttention の中ではやっておらず、モデルに渡す前で済ませる


- `Q @ K^T` の計算では、Q と K の前側の次元(batch, num_heads)が一致してないとダメ<br>
→ matmul のルール。`mha(x, x, x)` のように同じ入力から作る場合は自動で揃うので普段意識しない


- マスクテスト `mask = torch.tensor([[1,1,1,1,1,1,1,0,0,0], [1,1,1,1,1,0,0,0,0,0]])` の 1/0 という値そのものに意味はあるのか<br>
→ 1=本物、0=PAD という人間が決めた約束。`scores.masked_fill(mask == 0, float("-inf"))` で 0 の位置だけを -inf に置き換えてるので、判定してるのは「0 かどうか」のみ。本物側は 0 以外なら何でも動くが慣習で 1 を使う


- なぜ mask の形をわざわざ `None` で増やすのか<br>
→ mask は `(2, 10)`、scores は `(2, 8, 10, 10)`。形が違うので一緒に `masked_fill` できない。`(2, 1, 1, 10)` に整形すると、`1` の場所が scores の `8` と `10` に自動で広がって、形が一致する



## 学んだこと
- PyTorch のテンソル操作(matmul, softmax など)は 「最後の2次元 = 計算対象の行列、それより前 = 独立に並列処理されるバッチ次元」 バッチ次元は

- マルチヘッドアテンションは入力と出力の次元が同じであるため、transformerは6層や12層のエンコーダー、デコーダーを構成できる


- 文章0: 単語10個 × 各512次元<br>
文章1: 単語10個 × 各512次元<br>
        ↓ 縦に積む<br>
1つのテンソル shape = (2, 10, 512)<br>

- forward は明示的に呼び出さず、**インスタンス名の後ろに `()` をつけて引数を渡す書き方** で動く<br>
→ 例: `mha(x, x, x)` と書くと、`nn.Module` 内部で自動的に `forward(x, x, x)` が呼ばれる。


- ブロードキャスト<br>


形が違う2つのテンソルを、片方を自動で膨らませて計算できるようにする仕組み。

**ルール:**
- 形を右端から順に比べる(次元数が違う場合、足りない方は左に「1」が補われる)
- 各位置で「同じ数字」or「どちらかが1」なら成立。1 の方が複製されて相手に揃う
- それ以外(両方が異なる非1の数字)はエラー

**例:**

| a の形 | b の形 | 判定 | 結果の形 |
|---|---|---|---|
| `(2, 3)` | `(2, 3)` | 一致 → 複製不要 | `(2, 3)` |
| `(2, 3)` | `(1, 3)` | 1 が 2 に複製 | `(2, 3)` |
| `(2, 3)` | `(2, 1)` | 1 が 3 に複製 | `(2, 3)` |
| `(2, 3)` | `(3,)` | 左に1補完 → `(1, 3)` → 1 が 2 に複製 | `(2, 3)` |
| `(2, 3)` | `(3, 3)` | 2 ≠ 3、どちらも1じゃない | エラー |


a の形: (2, 3)        [[1, 2, 3],
                       [4, 5, 6]]

b の形: (3,)          [10, 20, 30]　→ (1, 3) [[10, 20, 30]] → (2, 3)  [[10, 20, 30],
         [10, 20, 30]]
 

a + b: [[11, 22, 33], 
        [14, 25, 36]]


